# 03_prepare_sampled_frames

This notebook extracts only the 1 fps annotation-aligned frames needed for the pilot experiment.

## Inputs
- `cholec80/splits/pilot_annotations_1fps.csv`
- original videos in `cholec80/videos/`
- compressed videos in:
  - `cholec80/videos_CRF18/`
  - `cholec80/videos_CRF23/`
  - `cholec80/videos_CRF28/`
  - `cholec80/videos_CRF35/`
  - `cholec80/videos_CRF51/`

## Outputs
Frames will be saved under:

- `cholec80/frames_sampled/orig/...`
- `cholec80/frames_sampled/CRF18/...`
- `cholec80/frames_sampled/CRF23/...`
- `cholec80/frames_sampled/CRF28/...`
- `cholec80/frames_sampled/CRF35/...`
- `cholec80/frames_sampled/CRF51/...`

Metadata tables will also be saved for each level.

## Naming convention
Each extracted image will use the raw frame index:

- `video01_000000.jpg`
- `video01_000025.jpg`
- `video01_000050.jpg`

This keeps filenames directly tied to the annotation table.

In [11]:
# ============================================================
# 1. Imports
# ============================================================
import json
import subprocess
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

In [12]:
# ============================================================
# 2. Paths
# ============================================================
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "cholec80"

VIDEOS_DIR = DATA_ROOT / "videos"
CRF_DIRS = {
    "orig": VIDEOS_DIR,
    "CRF18": DATA_ROOT / "videos_CRF18",
    "CRF23": DATA_ROOT / "videos_CRF23",
    "CRF28": DATA_ROOT / "videos_CRF28",
    "CRF35": DATA_ROOT / "videos_CRF35",
    "CRF51": DATA_ROOT / "videos_CRF51",
}

SPLITS_DIR = DATA_ROOT / "splits"
FRAMES_SAMPLED_DIR = DATA_ROOT / "frames_sampled"
OUTPUTS_DIR = DATA_ROOT / "outputs"
PREP_OUT_DIR = OUTPUTS_DIR / "frame_prep"

FRAMES_SAMPLED_DIR.mkdir(parents=True, exist_ok=True)
PREP_OUT_DIR.mkdir(parents=True, exist_ok=True)

for level in CRF_DIRS.keys():
    (FRAMES_SAMPLED_DIR / level).mkdir(parents=True, exist_ok=True)

In [13]:
 # ============================================================
# 3. Load pilot 1 fps annotations
# ============================================================
pilot_ann_1fps_path = SPLITS_DIR / "pilot_annotations_1fps.csv"
pilot_ann_1fps_df = pd.read_csv(pilot_ann_1fps_path)

print("Loaded:", pilot_ann_1fps_path)
print("Rows:", len(pilot_ann_1fps_df))
pilot_ann_1fps_df.head()

Loaded: /Users/niranjani/Desktop/video-compression-project/cholec80/splits/pilot_annotations_1fps.csv
Rows: 46099


,video,split,frame_idx,second_idx,phase_name,phase_id
0,video01,train,0,0,Preparation,0
1,video01,train,25,1,Preparation,0
2,video01,train,50,2,Preparation,0
3,video01,train,75,3,Preparation,0
4,video01,train,100,4,Preparation,0


In [14]:
# ============================================================
# 4. Quick sanity checks
# ============================================================
print("Splits:", sorted(pilot_ann_1fps_df["split"].unique()))
print("Videos:", pilot_ann_1fps_df["video"].nunique())
print("Min frame_idx:", pilot_ann_1fps_df["frame_idx"].min())
print("Max frame_idx:", pilot_ann_1fps_df["frame_idx"].max())

bad_rows = pilot_ann_1fps_df[pilot_ann_1fps_df["frame_idx"] % 25 != 0]
print("Rows violating 1 fps rule:", len(bad_rows))

Splits: ['test', 'train', 'val']
Videos: 20
Min frame_idx: 0
Max frame_idx: 145700
Rows violating 1 fps rule: 0


## Build a compact extraction table

We create one table that lists every frame we need to extract for every compression level.

In [15]:
# ============================================================
# 5. Build extraction metadata table
# ============================================================
levels = list(CRF_DIRS.keys())

all_extract_rows = []

for level in levels:
    for _, row in pilot_ann_1fps_df.iterrows():
        video = row["video"]
        split = row["split"]
        frame_idx = int(row["frame_idx"])
        second_idx = int(row["second_idx"])
        phase_name = row["phase_name"]
        phase_id = int(row["phase_id"])

        src_video_path = CRF_DIRS[level] / f"{video}.mp4"
        out_video_dir = FRAMES_SAMPLED_DIR / level / video
        out_video_dir.mkdir(parents=True, exist_ok=True)

        out_image_name = f"{video}_{frame_idx:06d}.jpg"
        out_image_path = out_video_dir / out_image_name

        all_extract_rows.append({
            "compression": level,
            "video": video,
            "split": split,
            "frame_idx": frame_idx,
            "second_idx": second_idx,
            "phase_name": phase_name,
            "phase_id": phase_id,
            "src_video_path": str(src_video_path),
            "out_image_path": str(out_image_path),
        })

extract_df = pd.DataFrame(all_extract_rows)

print("Total extraction rows:", len(extract_df))
extract_df.head()

Total extraction rows: 276594


,compression,video,split,frame_idx,second_idx,phase_name,phase_id,src_video_path,out_image_path
0,orig,video01,train,0,0,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...
1,orig,video01,train,25,1,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...
2,orig,video01,train,50,2,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...
3,orig,video01,train,75,3,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...
4,orig,video01,train,100,4,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...


In [16]:
# ============================================================
# 6. Save extraction plan
# ============================================================
extract_plan_path = PREP_OUT_DIR / "pilot_frame_extraction_plan.csv"
extract_df.to_csv(extract_plan_path, index=False)

print("Saved:", extract_plan_path)

Saved: /Users/niranjani/Desktop/video-compression-project/cholec80/outputs/frame_prep/pilot_frame_extraction_plan.csv


In [17]:
# ============================================================
# 7. Check source video existence
# ============================================================
missing_src = extract_df[~extract_df["src_video_path"].map(lambda p: Path(p).exists())]

print("Missing source video rows:", len(missing_src))
if len(missing_src) > 0:
    display(missing_src.head())

Missing source video rows: 0


## Extraction helper

We extract exact raw frame indices using ffmpeg's `select` filter.

Important:
- ffmpeg frame numbering is zero-based
- your `frame_idx` is also zero-based
- so we can directly use `eq(n\,frame_idx)`

In [18]:
# ============================================================
# 8. Fast 1 fps extraction helper (one ffmpeg call per video)
# ============================================================
def extract_video_at_1fps(src_video_path, out_video_dir, video_name, overwrite=False):
    src_video_path = Path(src_video_path)
    out_video_dir = Path(out_video_dir)
    out_video_dir.mkdir(parents=True, exist_ok=True)

    # output names: video01_000000.jpg, video01_000001.jpg, ...
    out_pattern = out_video_dir / f"{video_name}_%06d.jpg"

    # ffmpeg's image2 numbering starts at 1 by default.
    # We'll rename afterward so files become 0-based and match second_idx cleanly.
    existing_zero_based = sorted(out_video_dir.glob(f"{video_name}_*.jpg"))
    if len(existing_zero_based) > 0 and not overwrite:
        return "skipped_exists"

    if overwrite:
        for p in out_video_dir.glob(f"{video_name}_*.jpg"):
            p.unlink()

    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        "-i", str(src_video_path),
        "-vf", "fps=1",
        str(out_pattern),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        return f"error: {result.stderr.strip()}"

    # Rename 1-based output numbering to 0-based numbering
    produced = sorted(out_video_dir.glob(f"{video_name}_*.jpg"))
    if len(produced) == 0:
        return "error: no_frames_created"

    temp_pairs = []
    for p in produced:
        stem = p.stem
        idx_str = stem.split("_")[-1]
        one_based_idx = int(idx_str)
        zero_based_idx = one_based_idx - 1
        tmp_path = p.with_name(f"TMP_{video_name}_{zero_based_idx:06d}.jpg")
        p.rename(tmp_path)
        temp_pairs.append((tmp_path, zero_based_idx))

    for tmp_path, zero_based_idx in temp_pairs:
        final_path = out_video_dir / f"{video_name}_{zero_based_idx:06d}.jpg"
        tmp_path.rename(final_path)

    return "ok"

In [19]:
# ============================================================
# 9. Small test extraction (one full video at 1 fps)
# ============================================================
test_video = "video01"
test_level = "orig"

test_src_video_path = CRF_DIRS[test_level] / f"{test_video}.mp4"
test_out_video_dir = FRAMES_SAMPLED_DIR / test_level / test_video

print("Test source:", test_src_video_path)
print("Test out dir:", test_out_video_dir)

test_status = extract_video_at_1fps(
    src_video_path=test_src_video_path,
    out_video_dir=test_out_video_dir,
    video_name=test_video,
    overwrite=False,
)

print("Test extraction status:", test_status)

test_files = sorted(test_out_video_dir.glob(f"{test_video}_*.jpg"))
print("Num extracted test files:", len(test_files))
print("First 5:")
print([p.name for p in test_files[:5]])

Test source: /Users/niranjani/Desktop/video-compression-project/cholec80/videos/video01.mp4
Test out dir: /Users/niranjani/Desktop/video-compression-project/cholec80/frames_sampled/orig/video01
Test extraction status: ok
Num extracted test files: 1733
First 5:
['video01_000000.jpg', 'video01_000001.jpg', 'video01_000002.jpg', 'video01_000003.jpg', 'video01_000004.jpg']


## Full extraction

This may take a while, but it is much lighter than extracting full frame dumps.

In [20]:
# ============================================================
# 10. Run fast full extraction (per video, per level)
# ============================================================
OVERWRITE = False

video_level_rows = []

for level in levels:
    level_df = pilot_ann_1fps_df.copy()

    for video in tqdm(sorted(level_df["video"].unique()), desc=f"{level} videos"):
        src_video_path = CRF_DIRS[level] / f"{video}.mp4"
        out_video_dir = FRAMES_SAMPLED_DIR / level / video

        status = extract_video_at_1fps(
            src_video_path=src_video_path,
            out_video_dir=out_video_dir,
            video_name=video,
            overwrite=OVERWRITE,
        )

        video_level_rows.append({
            "compression": level,
            "video": video,
            "src_video_path": str(src_video_path),
            "out_video_dir": str(out_video_dir),
            "extract_status": status,
            "src_exists": src_video_path.exists(),
        })

video_extract_df = pd.DataFrame(video_level_rows)
video_extract_df

orig videos:   0%|          | 0/20 [00:00<?, ?it/s]

CRF18 videos:   0%|          | 0/20 [00:00<?, ?it/s]

CRF23 videos:   0%|          | 0/20 [00:00<?, ?it/s]

CRF28 videos:   0%|          | 0/20 [00:00<?, ?it/s]

CRF35 videos:   0%|          | 0/20 [00:00<?, ?it/s]

CRF51 videos:   0%|          | 0/20 [00:00<?, ?it/s]

,compression,video,src_video_path,out_video_dir,extract_status,src_exists
0,orig,video01,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...,skipped_exists,True
1,orig,video02,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...,ok,True
2,orig,video03,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...,ok,True
3,orig,video04,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...,ok,True
4,orig,video05,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...,ok,True
...,...,...,...,...,...,...
115,CRF51,video16,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...,ok,True
116,CRF51,video17,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...,ok,True
117,CRF51,video18,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...,ok,True
118,CRF51,video19,/Users/niranjani/Desktop/video-compression-pro...,/Users/niranjani/Desktop/video-compression-pro...,ok,True


In [21]:
# ============================================================
# 11. Video-level extraction summary
# ============================================================
video_extract_df["extract_status"].value_counts(dropna=False)

extract_status
ok                119
skipped_exists      1
Name: count, dtype: int64

In [23]:
# ============================================================
# 12. Build metadata from extracted files
# ============================================================
all_metadata_rows = []

for level in levels:
    level_ann_df = pilot_ann_1fps_df.copy()
    level_ann_df["compression"] = level

    for _, row in level_ann_df.iterrows():
        video = row["video"]
        split = row["split"]
        frame_idx = int(row["frame_idx"])
        second_idx = int(row["second_idx"])
        phase_name = row["phase_name"]
        phase_id = int(row["phase_id"])

        image_path = FRAMES_SAMPLED_DIR / level / video / f"{video}_{second_idx:06d}.jpg"

        all_metadata_rows.append({
            "compression": level,
            "video": video,
            "split": split,
            "frame_idx": frame_idx,
            "second_idx": second_idx,
            "phase_name": phase_name,
            "phase_id": phase_id,
            "image_path": str(image_path),
            "image_exists": image_path.exists(),
        })

combined_metadata_df = pd.DataFrame(all_metadata_rows)

print("Combined metadata rows:", len(combined_metadata_df))
combined_metadata_df.head(20)

Combined metadata rows: 276594


,compression,video,split,frame_idx,second_idx,phase_name,phase_id,image_path,image_exists
0,orig,video01,train,0,0,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
1,orig,video01,train,25,1,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
2,orig,video01,train,50,2,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
3,orig,video01,train,75,3,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
4,orig,video01,train,100,4,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
5,orig,video01,train,125,5,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
6,orig,video01,train,150,6,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
7,orig,video01,train,175,7,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
8,orig,video01,train,200,8,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
9,orig,video01,train,225,9,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True


In [24]:
# ============================================================
# 12. Missing-output check
# ============================================================
failed_df = combined_metadata_df[~combined_metadata_df["image_exists"]].copy()

print("Failed rows:", len(failed_df))
if len(failed_df) > 0:
    display(failed_df.head(20))

Failed rows: 120


,compression,video,split,frame_idx,second_idx,phase_name,phase_id,image_path,image_exists
1733,orig,video01,train,43325,1733,GallbladderRetraction,6,/Users/niranjani/Desktop/video-compression-pro...,False
4573,orig,video02,train,70975,2839,GallbladderRetraction,6,/Users/niranjani/Desktop/video-compression-pro...,False
10402,orig,video03,train,145700,5828,GallbladderRetraction,6,/Users/niranjani/Desktop/video-compression-pro...,False
11925,orig,video04,train,38050,1522,GallbladderRetraction,6,/Users/niranjani/Desktop/video-compression-pro...,False
14270,orig,video05,train,58600,2344,GallbladderRetraction,6,/Users/niranjani/Desktop/video-compression-pro...,False
16424,orig,video06,train,53825,2153,GallbladderRetraction,6,/Users/niranjani/Desktop/video-compression-pro...,False
20982,orig,video07,train,113925,4557,GallbladderRetraction,6,/Users/niranjani/Desktop/video-compression-pro...,False
22502,orig,video08,train,37975,1519,GallbladderRetraction,6,/Users/niranjani/Desktop/video-compression-pro...,False
25205,orig,video09,train,67550,2702,GallbladderRetraction,6,/Users/niranjani/Desktop/video-compression-pro...,False
26955,orig,video10,train,43725,1749,GallbladderRetraction,6,/Users/niranjani/Desktop/video-compression-pro...,False


In [33]:
def extract_exact_frame_fast(src_video_path, frame_idx, out_image_path, fps=25, overwrite=False):
    src_video_path = Path(src_video_path)
    out_image_path = Path(out_image_path)

    if out_image_path.exists() and not overwrite:
        return "skipped_exists"

    out_image_path.parent.mkdir(parents=True, exist_ok=True)

    # convert frame_idx → seconds
    timestamp = frame_idx / fps

    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        "-ss", str(timestamp),   # SEEK FIRST (fast)
        "-i", str(src_video_path),
        "-frames:v", "1",
        str(out_image_path),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        return f"error: {result.stderr.strip()}"
    if not out_image_path.exists():
        return "error: output_not_created"

    return "ok"

In [35]:
# ============================================================
# PATCH: extract only missing frames with fast seek
# ============================================================
patch_rows = []

for _, row in failed_df.iterrows():
    level = row["compression"]
    video = row["video"]
    frame_idx = int(row["frame_idx"])
    image_path = Path(row["image_path"])
    src_video_path = CRF_DIRS[level] / f"{video}.mp4"

    status = extract_exact_frame_fast(
        src_video_path=src_video_path,
        frame_idx=frame_idx,
        out_image_path=image_path,
        fps=25,
        overwrite=False,
    )

    patch_rows.append({
        "compression": level,
        "video": video,
        "frame_idx": frame_idx,
        "image_path": str(image_path),
        "patch_status": status,
        "image_exists_after_patch": image_path.exists(),
    })

patch_df = pd.DataFrame(patch_rows)
patch_df

,compression,video,frame_idx,image_path,patch_status,image_exists_after_patch
0,CRF23,video03,145700,/Users/niranjani/Desktop/video-compression-pro...,ok,True
1,CRF23,video04,38050,/Users/niranjani/Desktop/video-compression-pro...,ok,True
2,CRF23,video05,58600,/Users/niranjani/Desktop/video-compression-pro...,ok,True
3,CRF23,video06,53825,/Users/niranjani/Desktop/video-compression-pro...,ok,True
4,CRF23,video07,113925,/Users/niranjani/Desktop/video-compression-pro...,ok,True
...,...,...,...,...,...,...
73,CRF51,video16,73925,/Users/niranjani/Desktop/video-compression-pro...,ok,True
74,CRF51,video17,32600,/Users/niranjani/Desktop/video-compression-pro...,ok,True
75,CRF51,video18,48550,/Users/niranjani/Desktop/video-compression-pro...,ok,True
76,CRF51,video19,60600,/Users/niranjani/Desktop/video-compression-pro...,ok,True


In [36]:
# ============================================================
# Refresh image_exists after patch
# ============================================================
combined_metadata_df["image_exists"] = combined_metadata_df["image_path"].map(lambda p: Path(p).exists())

failed_df = combined_metadata_df[~combined_metadata_df["image_exists"]].copy()

print("Failed rows after patch:", len(failed_df))

Failed rows after patch: 0


## Save extraction logs and metadata

In [37]:
# ============================================================
# 13. Save video-level extraction log
# ============================================================
video_extract_log_path = PREP_OUT_DIR / "pilot_video_level_extraction_log.csv"
video_extract_df.to_csv(video_extract_log_path, index=False)

print("Saved:", video_extract_log_path)

Saved: /Users/niranjani/Desktop/video-compression-project/cholec80/outputs/frame_prep/pilot_video_level_extraction_log.csv


In [38]:
# ============================================================
# 14. Save clean metadata tables per compression level
# ============================================================
metadata_paths = {}

for level in levels:
    level_df = combined_metadata_df[combined_metadata_df["compression"] == level].copy()

    save_path = PREP_OUT_DIR / f"pilot_metadata_{level}.csv"
    level_df.to_csv(save_path, index=False)
    metadata_paths[level] = str(save_path)

print(json.dumps(metadata_paths, indent=2))

{
  "orig": "/Users/niranjani/Desktop/video-compression-project/cholec80/outputs/frame_prep/pilot_metadata_orig.csv",
  "CRF18": "/Users/niranjani/Desktop/video-compression-project/cholec80/outputs/frame_prep/pilot_metadata_CRF18.csv",
  "CRF23": "/Users/niranjani/Desktop/video-compression-project/cholec80/outputs/frame_prep/pilot_metadata_CRF23.csv",
  "CRF28": "/Users/niranjani/Desktop/video-compression-project/cholec80/outputs/frame_prep/pilot_metadata_CRF28.csv",
  "CRF35": "/Users/niranjani/Desktop/video-compression-project/cholec80/outputs/frame_prep/pilot_metadata_CRF35.csv",
  "CRF51": "/Users/niranjani/Desktop/video-compression-project/cholec80/outputs/frame_prep/pilot_metadata_CRF51.csv"
}


In [39]:
# ============================================================
# 15. Save combined metadata table
# ============================================================
combined_metadata_path = PREP_OUT_DIR / "pilot_metadata_all_levels.csv"
combined_metadata_df.to_csv(combined_metadata_path, index=False)

print("Saved:", combined_metadata_path)

Saved: /Users/niranjani/Desktop/video-compression-project/cholec80/outputs/frame_prep/pilot_metadata_all_levels.csv


In [40]:
# ============================================================
# 16. Count extracted frames by level
# ============================================================
count_by_level_df = (
    combined_metadata_df.groupby("compression")
    .agg(
        total_rows=("image_path", "size"),
        existing_images=("image_exists", "sum"),
    )
    .reset_index()
)

count_by_level_df

,compression,total_rows,existing_images
0,CRF18,46099,46099
1,CRF23,46099,46099
2,CRF28,46099,46099
3,CRF35,46099,46099
4,CRF51,46099,46099
5,orig,46099,46099


In [41]:
# ============================================================
# 17. Count extracted frames by level and split
# ============================================================
count_by_level_split_df = (
    combined_metadata_df.groupby(["compression", "split"])
    .agg(
        total_rows=("image_path", "size"),
        existing_images=("image_exists", "sum"),
    )
    .reset_index()
    .sort_values(["compression", "split"])
    .reset_index(drop=True)
)

count_by_level_split_df

,compression,split,total_rows,existing_images
0,CRF18,test,10081,10081
1,CRF18,train,26956,26956
2,CRF18,val,9062,9062
3,CRF23,test,10081,10081
4,CRF23,train,26956,26956
5,CRF23,val,9062,9062
6,CRF28,test,10081,10081
7,CRF28,train,26956,26956
8,CRF28,val,9062,9062
9,CRF35,test,10081,10081


In [42]:
# ============================================================
# 18. Preview a few saved paths
# ============================================================
combined_metadata_df.head(20)

,compression,video,split,frame_idx,second_idx,phase_name,phase_id,image_path,image_exists
0,orig,video01,train,0,0,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
1,orig,video01,train,25,1,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
2,orig,video01,train,50,2,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
3,orig,video01,train,75,3,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
4,orig,video01,train,100,4,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
5,orig,video01,train,125,5,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
6,orig,video01,train,150,6,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
7,orig,video01,train,175,7,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
8,orig,video01,train,200,8,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True
9,orig,video01,train,225,9,Preparation,0,/Users/niranjani/Desktop/video-compression-pro...,True


## Important note about filenames

Saved JPEG filenames are indexed by `second_idx`, not by raw `frame_idx`.

Example:
- `video01_000000.jpg` → second 0 → raw frame 0
- `video01_000001.jpg` → second 1 → raw frame 25
- `video01_000002.jpg` → second 2 → raw frame 50

The metadata CSV preserves both:
- `second_idx`
- `frame_idx`

So later notebooks should always use the metadata tables, not infer labels from filenames alone.

In [43]:
# ============================================================
# 19. Save prep summary JSON
# ============================================================
prep_summary = {
    "total_rows": int(len(combined_metadata_df)),
    "levels": levels,
    "video_level_status_counts": video_extract_df["extract_status"].value_counts(dropna=False).to_dict(),
    "count_by_level": count_by_level_df.to_dict(orient="records"),
    "count_by_level_split": count_by_level_split_df.to_dict(orient="records"),
    "combined_metadata_path": str(combined_metadata_path),
}

prep_summary_path = PREP_OUT_DIR / "pilot_frame_prep_summary.json"
with open(prep_summary_path, "w") as f:
    json.dump(prep_summary, f, indent=2)

print("Saved:", prep_summary_path)

Saved: /Users/niranjani/Desktop/video-compression-project/cholec80/outputs/frame_prep/pilot_frame_prep_summary.json


## Expected outputs

### Frame folders
- `cholec80/frames_sampled/orig/...`
- `cholec80/frames_sampled/CRF18/...`
- `cholec80/frames_sampled/CRF23/...`
- `cholec80/frames_sampled/CRF28/...`
- `cholec80/frames_sampled/CRF35/...`
- `cholec80/frames_sampled/CRF51/...`

### Metadata
In `cholec80/outputs/frame_prep/`:
- `pilot_frame_extraction_plan.csv`
- `pilot_video_level_extraction_log.csv`
- `pilot_metadata_orig.csv`
- `pilot_metadata_CRF18.csv`
- `pilot_metadata_CRF23.csv`
- `pilot_metadata_CRF28.csv`
- `pilot_metadata_CRF35.csv`
- `pilot_metadata_CRF51.csv`
- `pilot_metadata_all_levels.csv`
- `pilot_frame_prep_summary.json`